#### Get TSC

In [1]:
import pandas as pd
import os
import random
import ast
from NewsSentiment import TargetSentimentClassifier
from NewsSentiment.dataset import TooLongTextException
from openai import OpenAI
from tqdm.auto import tqdm
tqdm.pandas()

from dotenv import load_dotenv
load_dotenv()

gpt_api_key = os.environ.get('OPENAI_API_KEY')

tsc = TargetSentimentClassifier()

gpt_client = OpenAI(
    api_key=gpt_api_key
)

llama_client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

with open('prompt_EST.md', 'r', encoding='utf-8') as f:
    system_prompt_EST = f.read()

with open('prompt_OPP.md', 'r', encoding='utf-8') as f:
    system_prompt_OPP = f.read()

def get_datapoints(filepath:str):
    datapoints = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line.startswith("('") or line.startswith('("'):
                datapoints.append(ast.literal_eval(line))
    return datapoints

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
EST_datapoints_all = get_datapoints('../../NLP Pipeline/EST_annotation_samples.txt')

random.seed(42)
EST_datapoints = random.sample(EST_datapoints_all, 100)

In [3]:
human_EST_labels = ["negative", "positive", "neutral", "positive", "neutral", "neutral", "neutral", "neutral", "neutral", "neutral", "neutral", "negative", "neutral", "positive", "neutral", "negative", "negative", "neutral", "neutral", "negative", "neutral", "positive", "positive", "positive", "negative", "positive", "positive", "positive", "neutral", "negative", "neutral", "neutral", "positive", "negative", "neutral", "negative", "positive", "neutral", "neutral", "neutral", "neutral", "neutral", "neutral", "positive", "neutral", "negative", "neutral", "neutral", "neutral", "positive", "positive", "neutral", "positive", "neutral", "negative", "neutral", "positive", "positive", "negative", "positive", "negative", "neutral", "neutral", "neutral", "negative", "neutral", "negative", "positive", "neutral", "neutral", "negative", "neutral", "positive", "neutral", "neutral", "positive", "negative", "neutral", "negative", "neutral", "positive", "neutral", "negative", "positive", "positive", "neutral", "positive", "neutral", "neutral", "positive", "neutral", "neutral", "neutral", "negative", "positive", "positive", "negative", "positive", "neutral", "neutral"]

In [4]:
OPP_datapoints_all = get_datapoints('../../NLP Pipeline/OPP_annotation_samples.txt')

random.seed(42)
OPP_datapoints = random.sample(OPP_datapoints_all, 100)

In [5]:
human_OPP_labels = ["neutral", "positive", "negative", "neutral", "negative", "neutral", "neutral", "neutral", "neutral", "neutral", "neutral", "neutral", "neutral", "neutral", "negative", "neutral", "neutral", "negative", "neutral", "neutral", "positive", "negative", "positive", "negative", "neutral", "neutral", "neutral", "negative", "neutral", "negative", "neutral", "neutral", "neutral", "neutral", "negative", "neutral", "negative", "neutral", "neutral", "neutral", "negative", "neutral", "negative", "neutral", "neutral", "neutral", "neutral", "neutral", "negative", "neutral", "neutral", "neutral", "neutral", "neutral", "negative", "neutral", "neutral", "neutral", "negative", "negative", "positive", "neutral", "negative", "neutral", "neutral", "neutral", "positive", "neutral", "negative", "neutral", "neutral", "neutral", "negative", "positive", "negative", "neutral", "neutral", "negative", "neutral", "neutral", "negative", "neutral", "negative", "positive", "neutral", "neutral", "negative", "neutral", "neutral", "negative", "negative", "negative", "neutral", "positive", "positive", "neutral", "neutral", "neutral", "neutral", "negative"]

In [6]:
df_EST_annotations = pd.DataFrame({
    "Sentence": [datapoint[0] + datapoint[1] + datapoint[2] for datapoint in EST_datapoints],
    "Entity": [datapoint[1] for datapoint in EST_datapoints],
    "Human": human_EST_labels
})

df_EST_annotations

,Sentence,Entity,Human
0,"Keeping the RSS displeasure in mind, BJP chief...",RSS,negative
1,Narendra Modi swept the entire western belt by...,Narendra Modi,positive
2,the ruling BJP hoped to win 65-plus tally in t...,BJP,neutral
3,"From just two seats in 2014, BJP secured a thu...",BJP,positive
4,"In the last Lok Sabha elections, the seat was ...",Bharat Singh,neutral
...,...,...,...
95,BJP leaders said the party had identified such...,BJP,positive
96,"Mayawati said, ""Is this promise also a fake on...",BJP,negative
97,"By October 24, when results of the Maharashtra...",Devendra Fadnavis,positive
98,Congress President Rahul Gandhi with Andhra Pr...,N Chandrababu Naidu,neutral


In [7]:
df_OPP_annotations = pd.DataFrame({
    "Sentence": [datapoint[0] + datapoint[1] + datapoint[2] for datapoint in OPP_datapoints],
    "Entity": [datapoint[1] for datapoint in OPP_datapoints],
    "Human": human_OPP_labels
})

df_OPP_annotations

,Sentence,Entity,Human
0,"There were, however, clear indications from bo...",AAP,neutral
1,He also said Congress chief Rahul Gandhi has g...,Rahul Gandhi,positive
2,What an irony for the Congress that its only f...,Congress,negative
3,The occasion was a concert on Sunday night by ...,Ilayaraja,neutral
4,In Congress-ruled states of Rajasthan and Madh...,Congress,negative
...,...,...,...
95,While the Congress’s Imran Masood won Saharanp...,Imran Masood,neutral
96,"In other words, this is an electoral battle of...",Congress,neutral
97,Former Uttarakhand Chief Minister Harish Rawat...,Harish Rawat,neutral
98,(Express photo) CPI(M) general secretary Sitar...,Sitaram Yechury,neutral


In [8]:
GRU_TSC_EST_labels = []
for target in tqdm(EST_datapoints):
    try:
        s = tsc.infer(targets=[target], batch_size=1, disable_tqdm=True)
        labels = [result[0]['class_label'] for result in s]
        GRU_TSC_EST_labels.extend(labels)
    except TooLongTextException:
        continue

GRU_TSC_OPP_labels = []
for target in tqdm(OPP_datapoints):
    try:
        s = tsc.infer(targets=[target], batch_size=1, disable_tqdm=True)
        labels = [result[0]['class_label'] for result in s]
        GRU_TSC_OPP_labels.extend(labels)
    except TooLongTextException:
        continue

100%|██████████| 100/100 [00:19<00:00,  5.20it/s]


In [9]:
df_EST_annotations['NewsSentiment'] = GRU_TSC_EST_labels

df_EST_annotations

,Sentence,Entity,Human,NewsSentiment
0,"Keeping the RSS displeasure in mind, BJP chief...",RSS,negative,negative
1,Narendra Modi swept the entire western belt by...,Narendra Modi,positive,positive
2,the ruling BJP hoped to win 65-plus tally in t...,BJP,neutral,positive
3,"From just two seats in 2014, BJP secured a thu...",BJP,positive,positive
4,"In the last Lok Sabha elections, the seat was ...",Bharat Singh,neutral,positive
...,...,...,...,...
95,BJP leaders said the party had identified such...,BJP,positive,neutral
96,"Mayawati said, ""Is this promise also a fake on...",BJP,negative,negative
97,"By October 24, when results of the Maharashtra...",Devendra Fadnavis,positive,positive
98,Congress President Rahul Gandhi with Andhra Pr...,N Chandrababu Naidu,neutral,neutral


In [10]:
df_OPP_annotations['NewsSentiment'] = GRU_TSC_OPP_labels

df_OPP_annotations

,Sentence,Entity,Human,NewsSentiment
0,"There were, however, clear indications from bo...",AAP,neutral,neutral
1,He also said Congress chief Rahul Gandhi has g...,Rahul Gandhi,positive,positive
2,What an irony for the Congress that its only f...,Congress,negative,negative
3,The occasion was a concert on Sunday night by ...,Ilayaraja,neutral,positive
4,In Congress-ruled states of Rajasthan and Madh...,Congress,negative,neutral
...,...,...,...,...
95,While the Congress’s Imran Masood won Saharanp...,Imran Masood,neutral,positive
96,"In other words, this is an electoral battle of...",Congress,neutral,neutral
97,Former Uttarakhand Chief Minister Harish Rawat...,Harish Rawat,neutral,neutral
98,(Express photo) CPI(M) general secretary Sitar...,Sitaram Yechury,neutral,neutral


In [11]:
def LLM_annotate(datapoint, sys_prompt, model_name):
    left, entity, right = datapoint
    sentence = left + entity + right

    input = "Sentence: " + sentence + "\nEntity: " + entity

    if "gpt" in model_name:
        chat_completion = gpt_client.chat.completions.create(
            messages=[
                {
                    'role': 'system',
                    'content': sys_prompt,
                },
                {
                    'role': 'user',
                    'content': input,
                }
            ],
            model=model_name,
            seed=42, # temperature unavaliable for gpt5+ models
            reasoning_effort="none" # as per industry practice: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5
        )

    elif "llama" in model_name:
        chat_completion = llama_client.chat.completions.create(
            messages=[
                {
                    'role': 'system',
                    'content': sys_prompt,
                },
                {
                    'role': 'user',
                    'content': input,
                }
            ],
            model=model_name,
            temperature=0,
            reasoning_effort="none" # as per industry practice: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5
        )
        
        
    return chat_completion.choices[0].message.content

In [12]:
gpt_EST_labels = []
for datapoint in tqdm(EST_datapoints):
    gpt_EST_labels.append(LLM_annotate(datapoint, system_prompt_EST, "gpt-5.6-luna"))

gpt_OPP_labels = []
for datapoint in tqdm(OPP_datapoints):
    gpt_OPP_labels.append(LLM_annotate(datapoint, system_prompt_OPP, "gpt-5.6-luna"))

100%|██████████| 100/100 [01:40<00:00,  1.01s/it]


In [13]:
df_EST_annotations['ChatGPT'] = gpt_EST_labels

df_EST_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT
0,"Keeping the RSS displeasure in mind, BJP chief...",RSS,negative,negative,negative
1,Narendra Modi swept the entire western belt by...,Narendra Modi,positive,positive,positive
2,the ruling BJP hoped to win 65-plus tally in t...,BJP,neutral,positive,neutral
3,"From just two seats in 2014, BJP secured a thu...",BJP,positive,positive,positive
4,"In the last Lok Sabha elections, the seat was ...",Bharat Singh,neutral,positive,neutral
...,...,...,...,...,...
95,BJP leaders said the party had identified such...,BJP,positive,neutral,neutral
96,"Mayawati said, ""Is this promise also a fake on...",BJP,negative,negative,negative
97,"By October 24, when results of the Maharashtra...",Devendra Fadnavis,positive,positive,positive
98,Congress President Rahul Gandhi with Andhra Pr...,N Chandrababu Naidu,neutral,neutral,neutral


In [14]:
df_OPP_annotations['ChatGPT'] = gpt_OPP_labels

df_OPP_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT
0,"There were, however, clear indications from bo...",AAP,neutral,neutral,neutral
1,He also said Congress chief Rahul Gandhi has g...,Rahul Gandhi,positive,positive,positive
2,What an irony for the Congress that its only f...,Congress,negative,negative,negative
3,The occasion was a concert on Sunday night by ...,Ilayaraja,neutral,positive,neutral
4,In Congress-ruled states of Rajasthan and Madh...,Congress,negative,neutral,negative
...,...,...,...,...,...
95,While the Congress’s Imran Masood won Saharanp...,Imran Masood,neutral,positive,positive
96,"In other words, this is an electoral battle of...",Congress,neutral,neutral,neutral
97,Former Uttarakhand Chief Minister Harish Rawat...,Harish Rawat,neutral,neutral,neutral
98,(Express photo) CPI(M) general secretary Sitar...,Sitaram Yechury,neutral,neutral,neutral


In [15]:
llama_EST_labels = []
for datapoint in tqdm(EST_datapoints):
    llama_EST_labels.append(LLM_annotate(datapoint, system_prompt_EST, "llama3.1:8b"))

llama_OPP_labels = []
for datapoint in tqdm(OPP_datapoints):
    llama_OPP_labels.append(LLM_annotate(datapoint, system_prompt_OPP, "llama3.1:8b"))

100%|██████████| 100/100 [00:43<00:00,  2.31it/s]


In [16]:
df_EST_annotations['Llama'] = llama_EST_labels

df_EST_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
0,"Keeping the RSS displeasure in mind, BJP chief...",RSS,negative,negative,negative,negative
1,Narendra Modi swept the entire western belt by...,Narendra Modi,positive,positive,positive,positive
2,the ruling BJP hoped to win 65-plus tally in t...,BJP,neutral,positive,neutral,positive
3,"From just two seats in 2014, BJP secured a thu...",BJP,positive,positive,positive,positive
4,"In the last Lok Sabha elections, the seat was ...",Bharat Singh,neutral,positive,neutral,positive
...,...,...,...,...,...,...
95,BJP leaders said the party had identified such...,BJP,positive,neutral,neutral,neutral
96,"Mayawati said, ""Is this promise also a fake on...",BJP,negative,negative,negative,negative
97,"By October 24, when results of the Maharashtra...",Devendra Fadnavis,positive,positive,positive,positive
98,Congress President Rahul Gandhi with Andhra Pr...,N Chandrababu Naidu,neutral,neutral,neutral,neutral


In [17]:
df_OPP_annotations['Llama'] = llama_OPP_labels

df_OPP_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
0,"There were, however, clear indications from bo...",AAP,neutral,neutral,neutral,negative
1,He also said Congress chief Rahul Gandhi has g...,Rahul Gandhi,positive,positive,positive,negative
2,What an irony for the Congress that its only f...,Congress,negative,negative,negative,negative
3,The occasion was a concert on Sunday night by ...,Ilayaraja,neutral,positive,neutral,neutral
4,In Congress-ruled states of Rajasthan and Madh...,Congress,negative,neutral,negative,negative
...,...,...,...,...,...,...
95,While the Congress’s Imran Masood won Saharanp...,Imran Masood,neutral,positive,positive,negative
96,"In other words, this is an electoral battle of...",Congress,neutral,neutral,neutral,neutral
97,Former Uttarakhand Chief Minister Harish Rawat...,Harish Rawat,neutral,neutral,neutral,negative
98,(Express photo) CPI(M) general secretary Sitar...,Sitaram Yechury,neutral,neutral,neutral,neutral


In [18]:
df_EST_annotations.to_csv("EST_tone_annotation.csv", index=False)
df_OPP_annotations.to_csv("OPP_tone_annotation.csv", index=False)

### Results

In [19]:
import pandas as pd

df_EST_annotations = pd.read_csv("EST_tone_annotation.csv")
df_OPP_annotations = pd.read_csv("OPP_tone_annotation.csv")

EST

In [20]:
(df_EST_annotations["Human"] == df_EST_annotations["NewsSentiment"]).sum()

77

In [21]:
(df_EST_annotations["Human"] == df_EST_annotations["ChatGPT"]).sum()

78

In [22]:
(df_EST_annotations["Human"] == df_EST_annotations["Llama"]).sum()

56

OPP

In [23]:
(df_OPP_annotations["Human"] == df_OPP_annotations["NewsSentiment"]).sum()

84

In [24]:
(df_OPP_annotations["Human"] == df_OPP_annotations["ChatGPT"]).sum()

71

In [25]:
(df_OPP_annotations["Human"] == df_OPP_annotations["Llama"]).sum()

58

In [26]:
df_EST_annotations[df_EST_annotations["NewsSentiment"] != df_EST_annotations["ChatGPT"]]

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
2,the ruling BJP hoped to win 65-plus tally in t...,BJP,neutral,positive,neutral,positive
4,"In the last Lok Sabha elections, the seat was ...",Bharat Singh,neutral,positive,neutral,positive
6,The report submitted to PM Narendra Modi says ...,Narendra Modi,neutral,neutral,positive,positive
7,Anil Vij rose from an Akhil Bharatiya Vidyarth...,BJP,neutral,neutral,positive,positive
11,"But the “mandate” from May, as understood by s...",BJP,negative,neutral,negative,negative
13,The decision of the Congress and Samajwadi Par...,BJP,positive,positive,negative,positive
17,The outlier The BJP has been a key player in t...,BJP,neutral,positive,neutral,negative
24,He hit out at the BJP and said: “… has BJP MP ...,Meenakshi Lekhi,negative,neutral,negative,negative
26,Sharad Pawar (from twitter) (Sharad Pawar X) “...,Sharad Pawar,positive,positive,neutral,negative
34,Bharatiya Janata Party's (BJP) West Bengal in-...,Bharatiya Janata Party's,neutral,neutral,positive,positive


In [27]:
df_OPP_annotations[df_OPP_annotations["NewsSentiment"] != df_OPP_annotations["ChatGPT"]]

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
3,The occasion was a concert on Sunday night by ...,Ilayaraja,neutral,positive,neutral,neutral
4,In Congress-ruled states of Rajasthan and Madh...,Congress,negative,neutral,negative,negative
9,"Later, it became a pocket-borough of Mayawati'...",Mayawati,neutral,neutral,negative,negative
12,"Ridiculing the allegations against him, Mr. Ka...",Kannan,neutral,neutral,positive,negative
14,Chief minister Naveen Patnaik on Sunday resign...,Naveen Patnaik,negative,negative,neutral,neutral
15,At a time when the opposition parties are not ...,Omar Abdullah,neutral,neutral,positive,negative
16,Charanjeet Singh Rori defeated INC candidate A...,INC,neutral,neutral,negative,negative
20,Himanchal Pradesh CM Sukhvinder Singh Sukhu; K...,Meira Kumar,positive,positive,neutral,positive
35,"In 2014, Subhash Patel of BJP beat Ramesh Pate...",INC,neutral,neutral,negative,negative
38,From Rahul Gandhi to Vijay to Nadda: CJP invit...,Rahul Gandhi,neutral,neutral,positive,positive
